# Chapter 9: Design a Web Crawler
**From:** *System Design Interview*

---

## TL;DR

- A **web crawler** (spider) systematically downloads web pages by following links, used primarily for search engine indexing.
- Key design goals: **scalability** (billions of pages), **politeness** (per-host rate limits), **robustness** (handle traps/failures), and **extensibility** (plug-in new content types).
- The **URL Frontier** is the central data structure -- it manages both **priority queues** (crawl important pages first) and **politeness queues** (one thread per host with delays).
- **Content deduplication** via hashing eliminates the ~29% of duplicate web pages; **URL deduplication** via Bloom filters prevents infinite loops.
- Performance optimizations: **distributed crawl**, **DNS caching**, **geographic locality**, and **short timeouts**.

---
## Requirements

| Requirement | Value |
|---|---|
| Purpose | Search engine indexing |
| Pages / month | 1 billion |
| Content type | HTML only |
| Recrawl | Yes (new + edited pages) |
| Storage duration | 5 years |
| Duplicates | Ignore duplicate content |

---
## Back-of-the-Envelope Estimation

In [ ]:
# Web Crawler Estimation
pages_per_month = 1_000_000_000  # 1 billion
seconds_per_month = 30 * 24 * 3600

qps = pages_per_month / seconds_per_month
peak_qps = 2 * qps

avg_page_size_kb = 500
storage_per_month_tb = (pages_per_month * avg_page_size_kb) / (1e9)  # KB -> TB
storage_5_years_pb = storage_per_month_tb * 12 * 5 / 1000  # TB -> PB

print(f"Pages/month:          {pages_per_month:,.0f}")
print(f"QPS:                  {qps:,.0f} pages/sec")
print(f"Peak QPS:             {peak_qps:,.0f} pages/sec")
print(f"Storage/month:        {storage_per_month_tb:,.0f} TB")
print(f"Storage (5 years):    {storage_5_years_pb:,.0f} PB")

---
## High-Level Design

```mermaid
graph LR
    Seed[Seed URLs] --> Frontier[URL Frontier]
    Frontier --> Downloader[HTML Downloader]
    Downloader --> DNS[DNS Resolver]
    Downloader --> Parser[Content Parser]
    Parser --> ContentSeen{Content Seen?}
    ContentSeen -->|No| Extractor[URL Extractor]
    ContentSeen -->|Yes| Discard[Discard]
    Extractor --> Filter[URL Filter]
    Filter --> URLSeen{URL Seen?}
    URLSeen -->|No| Frontier
    URLSeen -->|Yes| Skip[Skip]
    Parser --> Storage[(Content Storage)]
```

### Component Responsibilities

| Component | Role |
|---|---|
| **Seed URLs** | Starting points for crawling (by locality or topic) |
| **URL Frontier** | Manages to-be-downloaded URLs with priority + politeness |
| **HTML Downloader** | Fetches pages via HTTP; respects robots.txt |
| **DNS Resolver** | Translates URLs to IP addresses (cached locally) |
| **Content Parser** | Validates HTML; runs as separate component to avoid slowing crawl |
| **Content Seen?** | Hash-based dedup to skip ~29% duplicate pages |
| **URL Extractor** | Parses links from HTML; converts relative to absolute URLs |
| **URL Filter** | Excludes blacklisted sites, error links, unwanted file types |
| **URL Seen?** | Bloom filter to avoid re-enqueuing known URLs |

---
## Deep Dive: URL Frontier

The URL Frontier is split into two modules:

```mermaid
graph TB
    subgraph Front Queues - Priority
        Prioritizer[Prioritizer] --> F1[Queue f1 - High Priority]
        Prioritizer --> F2[Queue f2 - Medium]
        Prioritizer --> FN[Queue fn - Low Priority]
    end
    subgraph Back Queues - Politeness
        Router[Queue Router] --> B1[Queue b1 - host-a.com]
        Router --> B2[Queue b2 - host-b.com]
        Router --> BN[Queue bn - host-c.com]
        B1 --> W1[Worker 1]
        B2 --> W2[Worker 2]
        BN --> WN[Worker N]
    end
    F1 --> Router
    F2 --> Router
    FN --> Router
```

- **Front queues** handle prioritization (PageRank, traffic, update frequency)
- **Back queues** enforce politeness (one queue per host, one worker per queue, configurable delay)
- **Hybrid storage**: most URLs on disk, with in-memory buffers for enqueue/dequeue

---
## Deep Dive: Performance Optimizations

| Optimization | How It Helps |
|---|---|
| **Distributed crawl** | Partition URL space across multiple servers with multiple threads each |
| **DNS caching** | Avoid 10-200ms synchronous DNS lookups per request |
| **Geographic locality** | Place crawl servers near target websites for faster downloads |
| **Short timeouts** | Skip unresponsive servers instead of blocking threads |
| **robots.txt caching** | Download and cache robots.txt periodically, not per-request |

## Deep Dive: Robustness

- **Consistent hashing** distributes load across downloaders; servers can be added/removed gracefully
- **Crawl state persistence** enables restart from saved checkpoints after failures
- **Exception handling** prevents individual errors from crashing the system
- **Data validation** guards against malformed content

## Deep Dive: Problematic Content

| Problem | Solution |
|---|---|
| **Redundant content** (~29%) | Hash/checksum fingerprinting |
| **Spider traps** (infinite loops) | Max URL length; manual blacklisting |
| **Data noise** (ads, spam) | Content quality filters |
| **Dynamic content** (JS/AJAX) | Server-side rendering before parsing |

---
## Trade-offs

| Decision | Option A | Option B | Recommendation |
|---|---|---|---|
| Traversal strategy | DFS | BFS | BFS -- DFS goes too deep; BFS explores breadth-first |
| URL storage | All in memory | All on disk | Hybrid -- memory buffer + disk for durability |
| Dedup strategy | Character comparison | Hash comparison | Hash -- O(1) comparison vs O(n) |
| Crawl approach | Single server | Distributed | Distributed -- single server cannot scale |
| Content types | HTML only | Extensible modules | Extensible -- plug in new downloaders as needed |

---
## Key Takeaways

1. **URL Frontier is the heart of the crawler.** It balances priority (crawl important pages first) with politeness (do not overwhelm any single host).

2. **Deduplication happens at two levels.** Content dedup avoids storing the same page twice; URL dedup avoids crawling the same link twice. Both use hash-based techniques.

3. **BFS over DFS.** Breadth-first search with priority weighting is the standard crawl strategy, but naive BFS must be augmented with politeness constraints.

4. **Distributed architecture is essential.** A single crawler server cannot handle billions of pages. Partition the URL space and use consistent hashing.

5. **Extensibility through modularity.** Design the system so new content types (images, PDFs) can be supported by plugging in new downloader modules.

6. **Robustness against the hostile web.** Spider traps, malformed HTML, unresponsive servers, and duplicate content are common -- the crawler must handle all gracefully.

---
## See Also

- [[url-frontier]] -- URL Frontier design with priority and politeness queues
- [[politeness-constraint]] -- Per-host rate limiting and crawl delay mechanisms
- [[content-deduplication]] -- Hash-based fingerprinting to detect duplicate pages
- [[url-deduplication]] -- Bloom filters for tracking visited URLs
- [[dns-caching]] -- DNS caching strategy for crawler performance
- [[crawler-extensibility]] -- Modular plug-in architecture for new content types